# Datamine APPEND Process Example

This notebook demonstrates how to configure and run the **`append`** process wrapper in `dmstudio`.

## Process Description

## APPEND

Append two or more files together.

The Data Definitions (DDs) of the files do not have to be the same; if they are not, then a superset APPEND takes place. APPEND may be in-place if the DDs of all the files are identical.

Both input files are optional. The operation is:-

1. If neither input file is defined, then APPEND will prompt for all files to be appended together. The first file specified defines the DD; all subsequent files must have an identical DD to or a subset of the first.
2. If only &IN1 or &IN2 is defined, then APPEND will copy the file to the output. If the input file is a catalogue file, then APPEND will append all the files in the catalogue into the output file. Only files with the same DD (or a subset of the DD) as the first file will be appended.
3. If both &IN1 and &IN2 files are specified, then the second file will be appended to the first. If &IN1 is a catalogue file, then all files specified in &IN1 will first be appended, followed by &IN2; or all files specified in &IN2, if &IN2 is a catalogue file. The output file DD will be a superset join of the two first file DDs, and all files on either input which match the DD (or have a subset of it) will be appended together. However if @PROTODD=1, then the first file in &IN1 will be taken as the definitive DD, and only files with this DD (or a subset of it) on either input will be appended together. This enables the first input to be used as a prototype for appending from a general catalogue.

In-place appending, where the output file is equal to &IN1, is allowed in the trivial case where two input files (not catalogues) are to be appended. In this case no retrieval criteria are permitted and both files must have identical DDs (or subset of &IN1). The output file can never be the same as &IN2.

If the optional parameter @SEQUENCE is set to 1, then a field 'FILENAME' is added to the output file. This field will contain the name of the file from which each record was appended. If @SEQUENCE is set to 2, then a field 'SEQUENCE' is added with a sequential file number (1,2,...) in it. If @SEQUENCE is set to greater than 2, then both fields are added. If APPEND is carried out in-place, then @SEQUENCE is ignored and a warning message displayed.

### Input Files:

* **in1** (Table):
  Input file 1. This may be a catalogue file. Omit for file prompting. Enter a prototype
  DD (and set PROTODD=1) for selection from a catalogue. Otherwise the DD of the first
  file will be combined with the first IN2 file (if any) for the output file DD, and only
  files matching (or a subset of) this DD will be appended.
  Required=No

* **in2** (Table):
  Input file 2. This may be a catalogue file. Omit for file prompting. Enter a catalogue
  file (and set PROTODD=1) for selection from this catalogue using the prototype on IN1.
  Otherwise the DD of the first file will be combined with the first IN1 entry (if any)
  for the output file DD, and only files matching (or a subset of) this DD will be
  appended. IN2 files are appended after IN1 files.
  Required=No

### Output Files:

* **out** (Table):
  Output file = file 1 for in place append, if IN1 nor IN2 are NOT catalogue files, both
  are defined, and have identical DDs. If SEQUENCE is set, then the output file will
  contain extra fields FILENAME (A,8) or SEQUENCE (N) or both.
  Required=Yes

### Fields:

### Parameters:

* **sequence**:

* **Options** (1,: add field FILENAME [A,8] into output file containing filenames of each):
  input file.; 2,: add field SEQUENCE [N] into output file containing a file sequence no.
  [1,2,...] for each file appended.; 3,: add both FILENAME and SEQUENCE fields.
  Range=0,3
  Values=0,1,2,3
  Default=0
  Required=No

* **protodd**:

* **Options** (1,: Use the file on IN1 as a prototype for selection of files from a catalogue):
  on IN2 to be appended. Ignored unless both IN1 and IN2 specified.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **print**:

* **Options** (1,: Show the output file DD after all files have been appended. If neither IN1):
  nor IN2 are specified, then the file names to be appended are prompted for:- Enter
  DATAMINE file name required, or <return> for default, or ] to end.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('append')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute append
print("Running append...")
dm_cmd.append(
    out_o='t_append_out',  # required
    # inmods_i=['optional'],  # optional
    # sequence_p=0,  # optional
    # protodd_p=0,  # optional
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("append execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_append_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")